In [21]:
import tensorflow as tf
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from tensorflow.keras.models import load_model
import pickle

In [22]:
## load the model
model = load_model("model.h5")

## read the picklefiles
with open("onehot_encoder_geo.pkl","rb") as file:
    onehot_encoder_geo = pickle.load(file)

with open("label_encoder_gender.pkl","rb") as file:
    label_encoder_gender= pickle.load(file)

with open("scaler.pkl","rb") as file:
    scaler = pickle.load(file)

In [35]:
## input data
input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

## modify the input
print(type(label_encoder_gender))
input_data['Gender']=label_encoder_gender.transform([input_data['Gender']])[0]
print(input_data)

## one hot encode the data
print(type(onehot_encoder_geo))
geo_encoded = onehot_encoder_geo.transform([[input_data['Geography']]]).toarray()
print(geo_encoded)
geo_encoded_df = pd.DataFrame(geo_encoded,columns = ["Geography_France","Geography_Germany","Geography_Spain"])
geo_encoded_df


<class 'sklearn.preprocessing._label.LabelEncoder'>
{'CreditScore': 600, 'Geography': 'France', 'Gender': 1, 'Age': 40, 'Tenure': 3, 'Balance': 60000, 'NumOfProducts': 2, 'HasCrCard': 1, 'IsActiveMember': 1, 'EstimatedSalary': 50000}
<class 'sklearn.preprocessing._encoders.OneHotEncoder'>
[[1. 0. 0.]]


C:\Users\ashis\AppData\Roaming\Python\Python312\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [47]:
## convert the input data into dataframe
input_df = pd.DataFrame([input_data])
input_df


,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,1,40,3,60000,2,1,1,50000


In [48]:
input_df = input_df.drop(columns=['Geography'],axis=1)
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,1,40,3,60000,2,1,1,50000


In [49]:
input_df = pd.concat([input_df,geo_encoded_df],axis=1)
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [55]:
input_df=scaler.transform(input_df)

C:\Users\ashis\AppData\Roaming\Python\Python312\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [56]:
input_df

array([[-6.7681861 ,  0.73902774, -3.68814088, -1.97778513, -1.21847468,
        -1.24715903, -0.11888692,  0.92443458, -1.74618097,  1.00450338,
        -1.91524949, -1.90861117]])

In [57]:
result = model.predict(input_df)
result[0][0]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step


0.002201286

In [59]:
if result[0][0]< 0.5:
    print("NO CHURN")
else:
    print("CHURN")


NO CHURN
